# 03 — Phase 4: Tensor Structuring & ML Export (Colab)

**Student 4 — Task 1 / Phase 4**

This notebook reads the aligned GeoTIFFs produced by Student 3 (Phase 3)
from Google Drive and converts them into ML-ready numpy tensors.

## What this notebook does
1. Mounts your Google Drive
2. Reads each fire from `aligned_compressed/<fire_id>/`
3. Stacks terrain + weather layers into feature tensors
4. Saves per-fire `.npz` tensors to Drive
5. Creates event-level train / val / test splits
6. Validates and visualises the results

## Before you start
- Make sure Student 3 has uploaded `aligned_compressed/` to your shared Google Drive folder.
- You only need to change the path in **Step 2** — everything else runs automatically.

---
### Feature tensor layout (15 channels per pixel)
| Index | Band | Source | Note |
|-------|------|--------|------|
| 0 | elevation_m | terrain tif | static |
| 1 | slope_deg | terrain tif | static |
| 2 | aspect_deg | terrain tif | static |
| 3–14 | 6 weather vars × (mean, max) | weather tif | aggregated across fire duration |

---
## Step 1 — Mount Google Drive and install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# rasterio is not pre-installed in Colab — install it once per session
!pip install rasterio -q
print('Packages ready.')

---
## Step 2 — Set your Google Drive paths

**This is the only cell you need to edit.**

Set `ALIGNED_DIR` to the folder where `aligned_compressed/` lives in your Drive.
If the folder is `My Drive/wildfire_project/data/aligned_compressed`, then set:
```python
ALIGNED_DIR = '/content/drive/MyDrive/wildfire_project/data/aligned_compressed'
```

In [ ]:
from pathlib import Path

# ── EDIT THIS ──────────────────────────────────────────────────────────────
ALIGNED_DIR = Path('/content/drive/MyDrive/wildfire_project/data/aligned_compressed')
# ───────────────────────────────────────────────────────────────────────────

# Outputs go next to the input data on Drive (so you don't lose them when Colab restarts)
DRIVE_DATA  = ALIGNED_DIR.parent          # …/data/
TENSOR_DIR  = DRIVE_DATA / 'tensors'
SPLITS_DIR  = DRIVE_DATA / 'splits'
REPORT_CSV  = ALIGNED_DIR / 'compression_report.csv'

TENSOR_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

print('aligned_compressed :', ALIGNED_DIR)
print('  exists            :', ALIGNED_DIR.exists())
print('tensors output      :', TENSOR_DIR)
print('splits  output      :', SPLITS_DIR)
print('report csv          :', REPORT_CSV)
print('  exists            :', REPORT_CSV.exists())

---
## Step 3 — Discover which fire events are ready

In [ ]:
import csv

def complete_fire_ids(report_csv: Path) -> list:
    """Return fire IDs where every row has fire_complete=True."""
    if not report_csv.exists():
        return []
    seen = {}
    with open(report_csv, newline='') as f:
        for row in csv.DictReader(f):
            fid = row['fire_id']
            ok  = row.get('fire_complete', '').strip().lower() == 'true'
            seen[fid] = seen.get(fid, True) and ok
    return [fid for fid, ok in seen.items() if ok]

fire_ids = complete_fire_ids(REPORT_CSV)
print(f'{len(fire_ids)} fire events ready for Phase 4.')
print('First 5:', fire_ids[:5])

In [ ]:
# Quick sanity check: verify expected files exist for each fire
REQUIRED = [
    'labels/burned_area_30m_aligned.tif',
    'labels/severity_30m_aligned.tif',
    'terrain/terrain_30m_aligned.tif',
    'weather/weather_stacked_30m_aligned.tif',
]

missing_count = 0
for fid in fire_ids:
    for layer in REQUIRED:
        p = ALIGNED_DIR / fid / layer
        if not p.exists():
            print(f'  MISSING: {fid}/{layer}')
            missing_count += 1

if missing_count == 0:
    print(f'All {len(fire_ids)} fires have the 4 required layers. Ready to proceed.')
else:
    print(f'WARNING: {missing_count} files missing — those fires will be skipped.')

---
## Step 4 — Core functions

Weather has a variable number of bands (6 variables × N days per fire).
We aggregate each variable to **mean** and **max** across all fire days,
giving a fixed 12-band weather summary per pixel regardless of fire duration.

> ⚠️ **Why streaming?** Some fires have 50+ days × 6 variables = 300+ bands.
> At 30 m resolution over a large fire that is **4–5 GB of data**.
> Loading it all at once crashes the Colab kernel silently (no error message —
> just a restart). The fix: read **one band at a time** and update running
> accumulators, so peak RAM stays under ~200 MB per fire no matter how
> long the fire burned.

In [ ]:
import gc
import json
import traceback

import numpy as np
import rasterio

N_WEATHER_VARS    = 6
WEATHER_VAR_NAMES = [
    'temperature_2m',
    'relative_humidity_pct',
    'wind_speed_ms',
    'wind_dir_deg',
    'surface_pressure',
    'total_precipitation',
]
TERRAIN_BAND_NAMES = ['elevation_m', 'slope_deg', 'aspect_deg']


def read_all_bands(path: Path) -> np.ndarray:
    """Read a small raster (labels, terrain) fully into RAM. Safe for ≤3 bands."""
    with rasterio.open(path) as src:
        return src.read()  # (C, H, W)


def aggregate_weather_streamed(weather_tif: Path) -> tuple:
    """
    Stream the weather GeoTIFF one band at a time.
    Computes mean and max per variable WITHOUT loading the full stack into RAM.

    Phase 2 band ordering — 6 variables per day, interleaved:
      band 1 = temperature_2m      day 1
      band 2 = relative_humidity   day 1
      band 3 = wind_speed_ms       day 1
      band 4 = wind_dir_deg        day 1
      band 5 = surface_pressure    day 1
      band 6 = total_precipitation day 1
      band 7 = temperature_2m      day 2  …

    Returns
    -------
    agg    : np.ndarray  shape (12, H, W)  float32
             layout: [var0_mean, var0_max, var1_mean, var1_max, …]
    n_days : int
    """
    with rasterio.open(weather_tif) as src:
        n_bands = src.count
        H, W    = src.height, src.width

        if n_bands % N_WEATHER_VARS != 0:
            raise ValueError(
                f'Weather band count {n_bands} is not divisible by {N_WEATHER_VARS}. '
                'Expected 6 variables per day.'
            )
        n_days = n_bands // N_WEATHER_VARS

        # float64 sum for numerically stable mean; float32 max
        var_sum = np.zeros((N_WEATHER_VARS, H, W), dtype='float64')
        var_max = np.full( (N_WEATHER_VARS, H, W), -np.inf, dtype='float32')

        for band_idx in range(1, n_bands + 1):        # rasterio bands are 1-indexed
            v    = (band_idx - 1) % N_WEATHER_VARS    # which variable (0–5)
            data = src.read(band_idx).astype('float32')
            var_sum[v] += data
            np.maximum(var_max[v], data, out=var_max[v])
            del data                                   # free immediately

    var_mean = (var_sum / n_days).astype('float32')
    del var_sum

    # Interleave mean and max: [var0_mean, var0_max, var1_mean, var1_max, …]
    agg = np.empty((N_WEATHER_VARS * 2, H, W), dtype='float32')
    for v in range(N_WEATHER_VARS):
        agg[v * 2]     = var_mean[v]
        agg[v * 2 + 1] = var_max[v]

    return agg, n_days          # (12, H, W),  int


def build_band_info() -> list:
    bands = []
    for name in TERRAIN_BAND_NAMES:
        bands.append({'name': name, 'source': 'terrain', 'aggregation': 'static'})
    for var in WEATHER_VAR_NAMES:
        bands.append({'name': f'{var}_mean', 'source': 'weather',
                      'aggregation': 'mean_over_fire_duration'})
        bands.append({'name': f'{var}_max',  'source': 'weather',
                      'aggregation': 'max_over_fire_duration'})
    return bands


def process_fire(fire_id: str) -> dict:
    fire_in  = ALIGNED_DIR / fire_id
    fire_out = TENSOR_DIR  / fire_id

    burned_tif   = fire_in / 'labels'  / 'burned_area_30m_aligned.tif'
    severity_tif = fire_in / 'labels'  / 'severity_30m_aligned.tif'
    terrain_tif  = fire_in / 'terrain' / 'terrain_30m_aligned.tif'
    weather_tif  = fire_in / 'weather' / 'weather_stacked_30m_aligned.tif'

    for p in [burned_tif, severity_tif, terrain_tif, weather_tif]:
        if not p.exists():
            return {'fire_id': fire_id, 'success': False, 'error': f'missing: {p.name}'}

    try:
        # Labels and terrain are small — load fully (≤3 bands each)
        burned   = read_all_bands(burned_tif)[0].astype('uint8')    # (H, W)
        severity = read_all_bands(severity_tif)[0].astype('uint8')  # (H, W)
        terrain  = read_all_bands(terrain_tif).astype('float32')    # (3, H, W)

        # Weather is large — stream it band-by-band to avoid OOM crash
        weather_agg, n_days = aggregate_weather_streamed(weather_tif)  # (12, H, W)

        # Stack all features: (15, H, W) → (H, W, 15)
        features_chw = np.concatenate([terrain, weather_agg], axis=0)   # (15, H, W)
        features     = np.moveaxis(features_chw, 0, -1)                  # (H, W, 15)
        del terrain, weather_agg, features_chw

        fire_out.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            fire_out / 'tensor.npz',
            features    = features,
            burned_area = burned,
            severity    = severity,
        )
        del features, burned, severity

        band_info = {
            'fire_id':         fire_id,
            'n_feature_bands': 15,
            'n_weather_days':  n_days,
            'bands':           build_band_info(),
        }
        (fire_out / 'band_info.json').write_text(json.dumps(band_info, indent=2))

        with rasterio.open(burned_tif) as src:
            h, w = src.height, src.width

        gc.collect()  # release memory before processing the next fire
        return {
            'fire_id': fire_id, 'success': True,
            'height': h, 'width': w,
            'n_feature_bands': 15,
            'n_weather_days': n_days, 'error': '',
        }

    except Exception as exc:
        traceback.print_exc()
        gc.collect()
        return {'fire_id': fire_id, 'success': False, 'error': str(exc)}


print('Core functions defined.')
print(f'  Band layout: {len(TERRAIN_BAND_NAMES)} terrain  +  {N_WEATHER_VARS*2} weather  =  15 total feature bands')

---
## Step 5 — Build tensors for all fires

This reads from Drive and writes back to Drive.
Each fire takes a few seconds. 39 fires ≈ 5–10 minutes total.

In [ ]:
results = []

for i, fire_id in enumerate(fire_ids, 1):
    r = process_fire(fire_id)
    results.append(r)
    status = 'OK  ' if r['success'] else 'FAIL'
    detail = (
        f"{r['height']}×{r['width']} px | {r['n_feature_bands']} feat bands | {r['n_weather_days']} weather days"
        if r['success'] else r['error']
    )
    print(f'[{i:02d}/{len(fire_ids)}] [{status}] {fire_id}  {detail}')

n_ok   = sum(r['success'] for r in results)
n_fail = len(results) - n_ok
print(f'\nFinished: {n_ok} succeeded, {n_fail} failed.')

In [ ]:
# Save a dataset manifest so build_splits and future tasks know what's available
manifest = {
    'n_fires_processed': len(results),
    'n_fires_success':   n_ok,
    'n_feature_bands':   len(build_band_info()),
    'feature_bands':     build_band_info(),
    'label_bands':       ['burned_area', 'severity'],
    'tensor_dir':        str(TENSOR_DIR),
    'fires':             results,
}
(TENSOR_DIR / 'dataset_manifest.json').write_text(json.dumps(manifest, indent=2))
print('Manifest saved to', TENSOR_DIR / 'dataset_manifest.json')

---
## Step 6 — Event-level train / val / test split

Split is done at the **fire event level** (not by pixel or patch).
Splitting by pixel would leak spatial information between splits and
make test results look better than they really are.

In [ ]:
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
TEST_FRAC  = 0.15
SEED       = 42

successful_ids = [r['fire_id'] for r in results if r['success']]
print(f'Splitting {len(successful_ids)} successful fires.')

rng = np.random.default_rng(SEED)
shuffled = list(successful_ids)
rng.shuffle(shuffled)

n       = len(shuffled)
n_train = max(1, round(n * TRAIN_FRAC))
n_val   = max(1, round(n * VAL_FRAC))
n_test  = n - n_train - n_val
if n_test < 1:
    n_val  = max(0, n_val - 1)
    n_test = 1

train_ids = shuffled[:n_train]
val_ids   = shuffled[n_train : n_train + n_val]
test_ids  = shuffled[n_train + n_val :]

print(f'  train : {len(train_ids)} fires  ({len(train_ids)/n:.0%})')
print(f'  val   : {len(val_ids)} fires  ({len(val_ids)/n:.0%})')
print(f'  test  : {len(test_ids)} fires  ({len(test_ids)/n:.0%})')

In [ ]:
# Save split files
for split_name, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    (SPLITS_DIR / f'{split_name}.txt').write_text('\n'.join(ids) + '\n')

splits_manifest = {
    'seed':      SEED,
    'fractions': {'train': TRAIN_FRAC, 'val': VAL_FRAC, 'test': TEST_FRAC},
    'counts':    {'train': len(train_ids), 'val': len(val_ids), 'test': len(test_ids)},
    'note':      'Split at fire-event level to prevent spatial leakage.',
    'train':     train_ids,
    'val':       val_ids,
    'test':      test_ids,
}
(SPLITS_DIR / 'splits.json').write_text(json.dumps(splits_manifest, indent=2))

print('Splits saved to', SPLITS_DIR)
print('  train.txt, val.txt, test.txt, splits.json')

---
## Step 7 — Validate: load a tensor and inspect it

In [ ]:
# Load the first successful tensor and print its properties
sample_id = successful_ids[0]
data = np.load(TENSOR_DIR / sample_id / 'tensor.npz')

X = data['features']       # (H, W, 15)
y_burned  = data['burned_area']  # (H, W)
y_severity = data['severity']    # (H, W)

print(f'Fire ID      : {sample_id}')
print(f'features     : shape={X.shape}  dtype={X.dtype}')
print(f'burned_area  : shape={y_burned.shape}  unique values={np.unique(y_burned)}')
print(f'severity     : shape={y_severity.shape}  unique values={np.unique(y_severity)}')
print(f'Burned pixels: {int(y_burned.sum())} / {y_burned.size}  ({y_burned.mean():.1%})')
print()

with open(TENSOR_DIR / sample_id / 'band_info.json') as f:
    band_info = json.load(f)

print('Feature band names:')
for i, b in enumerate(band_info['bands']):
    print(f'  [{i:2d}]  {b["name"]:35s}  mean={X[:,:,i].mean():.3f}  std={X[:,:,i].std():.3f}')

---
## Step 8 — Visualise one fire event

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(f'Fire {sample_id} — tensor channels', fontsize=13)

band_names = [b['name'] for b in band_info['bands']]

# Row 0: labels + terrain
axes[0, 0].imshow(y_burned,   cmap='Reds');  axes[0, 0].set_title('burned_area (label)')
axes[0, 1].imshow(y_severity, cmap='YlOrRd', vmin=0, vmax=4); axes[0, 1].set_title('severity (label)')
axes[0, 2].imshow(X[:, :, 0], cmap='terrain');  axes[0, 2].set_title(band_names[0])
axes[0, 3].imshow(X[:, :, 1], cmap='viridis');  axes[0, 3].set_title(band_names[1])

# Row 1: weather aggregations
for col, band_idx in enumerate([3, 5, 7, 9]):
    axes[1, col].imshow(X[:, :, band_idx], cmap='coolwarm')
    axes[1, col].set_title(band_names[band_idx])

for ax in axes.flat:
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## Step 9 — Dataset summary across all fires

In [ ]:
import pandas as pd

rows = []
for r in results:
    if not r['success']:
        continue
    npz = TENSOR_DIR / r['fire_id'] / 'tensor.npz'
    size_mb = round(npz.stat().st_size / 1024 / 1024, 2)
    split = (
        'train' if r['fire_id'] in train_ids else
        'val'   if r['fire_id'] in val_ids   else 'test'
    )
    rows.append({
        'fire_id':        r['fire_id'],
        'split':          split,
        'height':         r['height'],
        'width':          r['width'],
        'n_pixels':       r['height'] * r['width'],
        'weather_days':   r['n_weather_days'],
        'tensor_mb':      size_mb,
    })

df = pd.DataFrame(rows)
print('Total fires:', len(df))
print('Total pixels:', f"{df['n_pixels'].sum():,}")
print('Total tensor size:', f"{df['tensor_mb'].sum():.1f} MB")
print()
print(df.groupby('split')[['n_pixels', 'tensor_mb']].sum())
df.head(10)

---
## Done — Phase 4 complete

Your Google Drive now contains:

```
data/
├── aligned_compressed/      ← Phase 3 input (read-only)
├── tensors/
│   ├── dataset_manifest.json
│   └── <fire_id>/
│       ├── tensor.npz       ← features (H,W,15)  burned_area (H,W)  severity (H,W)
│       └── band_info.json
└── splits/
    ├── splits.json
    ├── train.txt
    ├── val.txt
    └── test.txt
```

### How to load a tensor in future notebooks
```python
import numpy as np
data = np.load('data/tensors/<fire_id>/tensor.npz')
X         = data['features']     # (H, W, 15) float32 — input features
y_burned  = data['burned_area']  # (H, W)     uint8   — binary label
y_severity = data['severity']    # (H, W)     uint8   — 4-class label
```

### Hand-off to Task 2 (modelling)
The modelling team should:
- Read fire IDs from `splits/train.txt`, `splits/val.txt`, `splits/test.txt`
- Load tensors for each fire from `tensors/<fire_id>/tensor.npz`
- Flatten `(H, W, 15)` → `(H*W, 15)` for tabular ML (Random Forest, XGBoost)
- Use spatial patches for CNN-based models
- Never mix train/val/test fires — splits are at the event level